<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/03_search/search_reranking_with_cross_encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Search Re-ranking with Cross-Encoders

## Objective
This notebook demonstrates a two-stage search pipeline where
semantic search is used for fast retrieval and a cross-encoder
is used to re-rank the top results for higher precision.

The goal is to understand why re-ranking improves search quality
in real-world NLP systems.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [ ]:
# Bi-encoder for retrieval
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Cross-encoder for re-ranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
query = "learn machine learning basics"

documents = [
    "Machine learning tutorials for beginners",
    "Deep learning concepts explained",
    "Natural language processing with transformers",
    "Python programming fundamentals",
    "Top tourist destinations in summer",
    "Football match highlights"
]

In [8]:
query_emb = bi_encoder.encode(query)
doc_embs = bi_encoder.encode(documents)

scores = cosine_similarity([query_emb], doc_embs)[0]

df = pd.DataFrame({
    "Document": documents,
    "Embedding Score": np.round(scores, 4)
}).sort_values(by="Embedding Score", ascending=False)

top_k = 3
candidates = df.head(top_k).copy()
candidates

,Document,Embedding Score
0,Machine learning tutorials for beginners,0.8806
1,Deep learning concepts explained,0.5567
3,Python programming fundamentals,0.3898


In [9]:
pairs = [[query, doc] for doc in candidates["Document"]]
rerank_scores = cross_encoder.predict(pairs)

candidates["Cross-Encoder Score"] = np.round(rerank_scores, 4)

candidates.sort_values(
    by="Cross-Encoder Score",
    ascending=False
)

,Document,Embedding Score,Cross-Encoder Score
0,Machine learning tutorials for beginners,0.8806,3.7356
1,Deep learning concepts explained,0.5567,-8.6288
3,Python programming fundamentals,0.3898,-10.3862


## Observations

- Embedding search retrieves relevant candidates quickly
- Cross-encoder re-ranking changes the order of top results
- Re-ranking improves precision at Rank-1

## Key Insight
Bi-encoders are best for recall, while cross-encoders are best
for precision. Combining both yields high-quality search systems.